# P6 magnitude v2 — does the global-RNG leak move a real success rate?

**v2, 2026-09-23.** Two changes from v1, both forced by Mohan's live run:
(a) the numpy pin is applied LAST and with `--no-deps`, because in v1 every later install
walked over it and the run reported numpy 2.0.2; (b) a **stack-drift control** (cell 1b)
reproduces v6's exact procedure so there is an apples-to-apples test of whether the drift
mattered. v1's Stage 1 hashes were never comparable to the 09-13/09-21 values.


**Pre-registration. Written before the run. Do not edit after seeing results.**

The P6 determinism gate (`robosuite_gate_v6.ipynb`) proved the leak exists as a **bit-identity**:
on the raw reset path, consuming `np.random` between episodes changes the next episode's initial
state (T5 fails 9/9, 27 distinct hashes). It did **not** measure what that costs a policy, and both
papers say so explicitly. This notebook measures it.

## Prediction, fixed now

1. **Stage 1 (no policy):** arm A (raw reset + RNG burn) and arm B (per-episode reseed) will
   produce **disjoint** initial-state hash sets. Overlap = 0 / N. *This re-confirms v6 and is the
   control: if overlap is not 0, everything below is uninterpretable and the notebook says
   INCONCLUSIVE.*
2. **Stage 2 (policy):** the two arms will differ in success rate. **I do not predict a direction
   or a magnitude.** The arms are scored on different episodes, so any difference is a
   benchmark-identity difference, not a policy difference. The number this produces is
   "how far a success rate can move for this reason", nothing more.
3. **Paired per-episode agreement** between arms will be **low**, because episode *i* of arm A and
   episode *i* of arm B are not the same episode.

## What would falsify the paper's framing

If overlap is 0 and the success rates are nonetheless identical to within noise, then the leak is
real but costs nothing measurable at this N, and **the paper should say that**. That is a real
possible outcome and it is not a failed experiment.

## Honest limits, stated before running

* One policy, one suite, small N. This is a magnitude *probe*, not an estimate with a CI.
* SmolVLA's model card reports **no LIBERO success rate**, so there is no external anchor for the
  absolute number. Only the A-vs-B difference is interpretable.
* **This notebook has never been executed.** It was written on a machine with no GPU and no Kaggle
  access. Expect to fix something on the first run; the stages are separated so a late failure
  still leaves you with Stage 1's answer.


In [ ]:
# ---- cell 0: pre-registration constants. Change nothing here after the first run. ----
PREREG = {
    "written_utc": "2026-09-22",
    "prediction_1_overlap": "arm A and arm B initial-state hash sets are DISJOINT (overlap 0)",
    "prediction_2_success": "arms differ; NO direction or magnitude predicted",
    "prediction_3_paired":  "per-episode paired agreement between arms is LOW",
    "falsifier": "overlap==0 but success rates identical -> leak real, cost not measurable at this N",
}

SUITE         = "libero_object"   # one suite; task 0 first
TASK_ID       = 0
N_EPISODES    = 20                # per arm. raise only if Stage 2 finishes fast
MAX_STEPS     = 300               # per episode
POLICY_REPO   = "HuggingFaceVLA/smolvla_libero"   # verified public + apache-2.0 via HF API 2026-09-22
BURN_DRAWS    = 1000              # RNG draws between episodes in arm A, emulating a sampling policy
SEED_BASE     = 10000
PROGRESS_EVERY = 10               # partial JSON written this often, so a timeout still yields a number

import json, os, sys, time, hashlib, platform
print(json.dumps(PREREG, indent=2))
print("\nSUITE=%s TASK=%d N=%d MAXSTEPS=%d POLICY=%s" % (SUITE, TASK_ID, N_EPISODES, MAX_STEPS, POLICY_REPO))
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("output dir:", OUT)

## 1 · Environment. Pinned. ~10 min on a cold Kaggle session.

In [ ]:
import subprocess, sys, os, time
def sh(cmd, check=True, tail=12):
    print(">>>", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout.strip().splitlines()
    print("\n".join(out[-tail:]), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError("FAILED (%d): %s" % (r.returncode, cmd))
    return r.returncode

t0 = time.time()
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo NO-GPU", check=False)
print("python:", sys.version.split()[0])

# ---------------------------------------------------------------------------
# v2 INSTALL ORDER. v1 pinned numpy FIRST and every later install walked over it:
# the 2026-09-23 Kaggle run reported numpy 2.0.2 despite 'numpy==1.26.4' in cell 1.
# robosuite, opencv, torch and transformers all declare numpy>=2 compatible ranges
# and pip happily upgrades. So: install everything that wants numpy FIRST, take
# opencv without its dependency chain, and pin numpy LAST so nothing can move it.
# ---------------------------------------------------------------------------
sh("pip install -q 'robosuite==1.4.0' 'bddl==1.0.1' 'gym==0.25.2' 'mujoco<3.2' easydict")
sh("pip install -q --no-deps opencv-python-headless")
sh("pip install -q 'transformers>=4.50' accelerate safetensors huggingface_hub")
sh("pip install -q --force-reinstall --no-deps 'numpy==1.26.4'")   # LAST, and --no-deps

import importlib
importlib.invalidate_caches()
import numpy as _np
print("\nnumpy after pinning:", _np.__version__)

NUMPY_OK = _np.__version__.startswith("1.26.4")
if not NUMPY_OK:
    print("\n" + "="*70)
    print("NUMPY PIN STILL LOST -> %s, wanted 1.26.4" % _np.__version__)
    print("Do NOT treat the magnitude number as comparable to the 09-13/09-21 run.")
    print("Run the stack-drift control below anyway: if it reproduces the six")
    print("reference hashes, the drift is harmless and the run still counts.")
    print("="*70)
else:
    print("numpy pin held.")

if not os.path.isdir("/kaggle/working/LIBERO"):
    sh("git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git /kaggle/working/LIBERO")
sys.path.insert(0, "/kaggle/working/LIBERO")
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
print("env setup %.1f min" % ((time.time()-t0)/60))

In [ ]:
# record the exact stack, so the JSON is self-describing
import importlib, json, platform
VERS = {"python": platform.python_version()}
for m in ("numpy","robosuite","mujoco","gym","torch","transformers"):
    try:
        VERS[m] = importlib.import_module(m).__version__
    except Exception as e:
        VERS[m] = "ABSENT (%s)" % type(e).__name__
print(json.dumps(VERS, indent=2))

## 1b · Stack-drift control — **the cell that decides whether this run counts**

🔴 **Read this before interpreting anything.**

The Stage 1 hashes below are **not** comparable to the 09-13 / 09-21 reference values. Those came
from `robosuite_gate_v6.ipynb`, which uses different seeds (11,12,13 vs 10000+), a different
camera size (128 vs 256), and a different hash function (`env.get_sim_state()` vs
`sim.get_state().flatten()`). Comparing them would be meaningless.

So this cell **reproduces v6's procedure exactly** on `libero_object` and compares against the six
hashes v5 produced and v6 reproduced. That is the only apples-to-apples test of whether the stack
drift (numpy 2.0.2, torch 2.10.0, transformers 5.0.0) changed the simulator's behaviour.

**The rule, fixed before the result is seen:**

* **All six match →** the stack drift is **harmless**. The magnitude number from Stage 2 stands
  and is comparable to the paper's other results.
* **Any mismatch →** **INCONCLUSIVE-ON-CONTROL.** The magnitude number is not comparable and must
  not go in the paper. Rerun with the v2 install order in cell 1 and check this cell again.

In [ ]:
# v6's exact procedure on libero_object, for comparison against known values.
import numpy as np, hashlib, json, os, sys, time
sys.path.insert(0, "/kaggle/working/LIBERO")
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

# v5's libero_object hashes, reproduced by v6 on 2026-09-13 and again by Mohan on 2026-09-21.
REFERENCE = {
    "T2":        {11: "cbb3470b4c447f68", 12: "c732927029d36125", 13: "c09097460e7814a5"},
    "T5_burned": {11: "e78e0d1a180b3e6a", 12: "42bec853db81b54b", 13: "dfa9091d9724f85f"},
}
K_SHORT, K_LONG, GLOBAL_DRAWS = 40, 80, 1000

def _bddl(suite="libero_object"):
    b = benchmark.get_benchmark_dict()[suite]()
    t = b.get_task(0)
    return os.path.join(get_libero_path("bddl_files"), t.problem_folder, t.bddl_file)

def _key(env):                       # v6's hash, not Stage 1's
    s = np.asarray(env.get_sim_state(), dtype=np.float64)
    return hashlib.sha256(np.ascontiguousarray(s).tobytes()).hexdigest()[:16]

def _acts(k, tag, env):
    rs = np.random.RandomState(1234 if tag == "armA" else 5678)
    return [rs.uniform(-0.2, 0.2, size=env.env.action_dim) for _ in range(k)]

def _roll(bddl, s, k, tag, burn=0):  # v6's roll_reset, reseed=None path
    e = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=128, camera_widths=128)
    e.seed(int(s)); e.reset()
    for a in _acts(k, tag, e):
        e.step(a)
    for _ in range(burn):
        np.random.random()
    e.reset()
    kk = _key(e); e.close(); return kk

t0 = time.time(); bddl = _bddl(); got = {"T2": {}, "T5_burned": {}}
for s in (11, 12, 13):
    got["T2"][s]        = _roll(bddl, s, K_LONG,  "armA", burn=0)
    got["T5_burned"][s] = _roll(bddl, s, K_SHORT, "armA", burn=GLOBAL_DRAWS)

matches, total = 0, 0
print("%-12s %-5s %-18s %-18s %s" % ("test", "seed", "reference", "this run", ""))
for test in ("T2", "T5_burned"):
    for s in (11, 12, 13):
        ref, cur = REFERENCE[test][s], got[test][s]
        ok = (ref == cur); matches += ok; total += 1
        print("%-12s %-5d %-18s %-18s %s" % (test, s, ref, cur, "MATCH" if ok else "*** DIFFERS ***"))

DRIFT_HARMLESS = (matches == total)
CONTROL_VERDICT = ("STACK DRIFT HARMLESS (%d/%d reference hashes reproduced) -- "
                   "the magnitude number stands" % (matches, total)) if DRIFT_HARMLESS else \
                  ("INCONCLUSIVE-ON-CONTROL (%d/%d) -- the magnitude number is NOT comparable; "
                   "rerun with the v2 install order" % (matches, total))
print("\nnumpy:", np.__version__, "| control took %.1f min" % ((time.time()-t0)/60))
print("VERDICT:", CONTROL_VERDICT)

json.dump({"numpy": np.__version__, "reference": REFERENCE, "observed": got,
           "matches": matches, "total": total, "drift_harmless": DRIFT_HARMLESS,
           "verdict": CONTROL_VERDICT},
          open(os.path.join(OUT, "p6_stackdrift_control.json"), "w"), indent=2)

## 2 · Stage 1 — the control, and the core measurement. No policy. Minutes.

Two arms, identical seed lists.

* **Arm A (leak present):** the shipped raw path. `env.reset()` with the seed set once, then
  `BURN_DRAWS` draws from the global `np.random` between episodes, emulating a policy that samples.
* **Arm B (fixed):** `env.seed(s); env.reset()` per episode.

If the leak is real, A's initial states drift and B's do not.

In [ ]:
import numpy as np, hashlib, json, time
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

def make_env(suite, task_id):
    bench = benchmark.get_benchmark_dict()[suite]()
    task  = bench.get_task(task_id)
    bddl  = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env   = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    return env, task

def state_hash(env):
    return hashlib.sha256(np.asarray(env.env.sim.get_state().flatten(), dtype=np.float64).tobytes()).hexdigest()[:16]

SEEDS = [SEED_BASE + i for i in range(N_EPISODES)]

def init_states(arm):
    env, task = make_env(SUITE, TASK_ID)
    hashes = []
    try:
        if arm == "A":
            np.random.seed(SEEDS[0]); env.seed(SEEDS[0])
        for i, s in enumerate(SEEDS):
            if arm == "B":
                np.random.seed(s); env.seed(s)
            env.reset()
            hashes.append(state_hash(env))
            if arm == "A":
                _ = np.random.rand(BURN_DRAWS)   # the leak: unrelated consumption
    finally:
        env.close()
    return hashes, task.language

t0 = time.time()
hA, lang = init_states("A")
hB, _    = init_states("B")
overlap  = len(set(hA) & set(hB))
stage1 = {"task_language": lang, "seeds": SEEDS,
          "armA_initstate_hashes": hA, "armB_initstate_hashes": hB,
          "overlap": overlap, "armA_distinct": len(set(hA)), "armB_distinct": len(set(hB)),
          "minutes": round((time.time()-t0)/60, 2)}
print(json.dumps({k:v for k,v in stage1.items() if "hashes" not in k}, indent=2))
print("\narm A first 5:", hA[:5]); print("arm B first 5:", hB[:5])
VERDICT_S1 = "LEAK CONFIRMED (arms disjoint)" if overlap == 0 else "INCONCLUSIVE — arms overlap; Stage 2 is not interpretable"
print("\nSTAGE 1:", VERDICT_S1)
json.dump({"prereg":PREREG,"versions":VERS,"stage1":stage1,"verdict_stage1":VERDICT_S1},
          open(os.path.join(OUT,"p6_magnitude_partial.json"),"w"), indent=2)

## 3 · Stage 2 — success rate per arm, with the policy. Hours.

Loads SmolVLA and rolls out both arms on the **same seed list**. Writes partial JSON every
`PROGRESS_EVERY` episodes, so a session timeout still leaves a usable number.

If the policy will not load, **stop here and report Stage 1** — that is still the paper's claim,
re-confirmed. Do not improvise a different policy mid-run; that would break the pre-registration.

In [ ]:
# Policy load. Isolated so a failure here does not cost Stage 1.
POLICY_OK, policy, pol_err = False, None, None
try:
    sh("pip install -q 'lerobot'", check=True)
    from lerobot.common.policies.factory import make_policy  # path may differ by lerobot version
    import torch
    policy = make_policy(POLICY_REPO)
    policy.eval().to("cuda")
    POLICY_OK = True
    print("policy loaded:", POLICY_REPO)
except Exception as e:
    pol_err = "%s: %s" % (type(e).__name__, e)
    print("POLICY LOAD FAILED ->", pol_err)
    print("\nThis is the most likely failure point and it is isolated on purpose.")
    print("If lerobot's API moved, fix the two lines above; everything else is unaffected.")
    print("Stage 1 above is already saved to p6_magnitude_partial.json.")

In [ ]:
import numpy as np, json, time, hashlib

def obs_to_policy(obs, device="cuda"):
    """LIBERO obs -> the keys this checkpoint declares (verified from its config.json):
       observation.images.image, observation.images.image2, observation.state."""
    import torch
    img  = torch.from_numpy(obs["agentview_image"][::-1].copy()).permute(2,0,1)[None].float()/255.
    img2 = torch.from_numpy(obs["robot0_eye_in_hand_image"][::-1].copy()).permute(2,0,1)[None].float()/255.
    st   = np.concatenate([obs["robot0_eef_pos"], obs["robot0_eef_quat"], obs["robot0_gripper_qpos"]])
    return {"observation.images.image":  img.to(device),
            "observation.images.image2": img2.to(device),
            "observation.state": torch.from_numpy(st)[None].float().to(device),
            "task": [lang]}

def rollout(arm):
    env, task = make_env(SUITE, TASK_ID)
    results = []
    try:
        if arm == "A":
            np.random.seed(SEEDS[0]); env.seed(SEEDS[0])
        for i, s in enumerate(SEEDS):
            if arm == "B":
                np.random.seed(s); env.seed(s)
            obs = env.reset()
            h0  = state_hash(env)
            if hasattr(policy, "reset"): policy.reset()
            done, steps, success = False, 0, False
            while not done and steps < MAX_STEPS:
                import torch
                with torch.no_grad():
                    a = policy.select_action(obs_to_policy(obs)).cpu().numpy().flatten()[:7]
                obs, _, done, info = env.step(a.tolist())
                steps += 1
                success = bool(env.check_success()) if hasattr(env, "check_success") else bool(done)
                if success: break
            results.append({"i": i, "seed": s, "init_hash": h0, "success": success, "steps": steps})
            if arm == "A":
                _ = np.random.rand(BURN_DRAWS)
            if (i+1) % PROGRESS_EVERY == 0:
                sr = sum(r["success"] for r in results)/len(results)
                print("  arm %s  %d/%d  success so far %.2f" % (arm, i+1, len(SEEDS), sr), flush=True)
                json.dump({"prereg":PREREG,"versions":VERS,"stage1":stage1,
                           "partial_arm":arm,"partial_results":results},
                          open(os.path.join(OUT,"p6_magnitude_partial.json"),"w"), indent=2)
    finally:
        env.close()
    return results

if POLICY_OK:
    t0 = time.time()
    rA = rollout("A"); rB = rollout("B")
    sA = sum(r["success"] for r in rA); sB = sum(r["success"] for r in rB)
    paired_agree = sum(1 for a,b in zip(rA,rB) if a["success"] == b["success"])
    stage2 = {
        "armA_successes": sA, "armB_successes": sB, "n": len(SEEDS),
        "armA_rate": sA/len(SEEDS), "armB_rate": sB/len(SEEDS),
        "delta_pp": (sB-sA)/len(SEEDS)*100.0,
        "paired_agreement": paired_agree, "paired_agreement_frac": paired_agree/len(SEEDS),
        "init_hash_overlap_stage2": len(set(r["init_hash"] for r in rA) & set(r["init_hash"] for r in rB)),
        "hours": round((time.time()-t0)/3600, 2),
        "armA": rA, "armB": rB,
    }
    print(json.dumps({k:v for k,v in stage2.items() if k not in ("armA","armB")}, indent=2))
else:
    stage2 = {"SKIPPED": "policy did not load", "error": pol_err}
    print("Stage 2 skipped.")

In [ ]:
# final JSON — this is the file to download from Kaggle output
final = {"prereg": PREREG, "versions": VERS, "config": {
            "suite": SUITE, "task_id": TASK_ID, "n_episodes": N_EPISODES,
            "max_steps": MAX_STEPS, "policy": POLICY_REPO, "burn_draws": BURN_DRAWS,
            "seed_base": SEED_BASE},
         "stage1": stage1, "verdict_stage1": VERDICT_S1, "stage2": stage2,
         "stack_drift_control": {"verdict": CONTROL_VERDICT, "drift_harmless": DRIFT_HARMLESS, "numpy": VERS.get("numpy")}}
path = os.path.join(OUT, "p6_magnitude_result.json")
json.dump(final, open(path, "w"), indent=2)
print("WROTE", path)
print("\n================ READ THIS BACK TO THE VAULT ================")
print("stack-drift control  :", CONTROL_VERDICT)
print("stage 1 overlap      :", stage1["overlap"], "( 0 = leak confirmed )")
if POLICY_OK:
    print("arm A success        : %d/%d" % (stage2["armA_successes"], stage2["n"]))
    print("arm B success        : %d/%d" % (stage2["armB_successes"], stage2["n"]))
    print("delta (B - A)        : %+.1f pp" % stage2["delta_pp"])
    print("paired agreement     : %d/%d" % (stage2["paired_agreement"], stage2["n"]))
    print("wall clock (stage 2) : %.2f h" % stage2["hours"])
else:
    print("stage 2              : SKIPPED —", stage2.get("error"))
print("============================================================")